<a href="https://colab.research.google.com/github/JoieLiu/PL-repo/blob/main/%E3%80%8CHW2_part2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_gradio_ipynb%E3%80%8D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 安裝必要套件
!pip install -q google-genai gspread gradio

import gradio as gr
import pandas as pd
import gspread
import requests
import json
import os
from datetime import datetime
from google.colab import auth, userdata
from google.auth import default

# --- 初始化設定 ---
SHEET_URL = "https://docs.google.com/spreadsheets/d/1Nm4gOSDMPGRn-PJdPwSYLwY-J-aGyCs46XECwzxiqk8/edit?usp=sharing"
WORKSHEET_NAME = "工作表1"

def get_sheet():
    """驗證並取得 Google Sheet 工作表物件"""
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)
    sh = gc.open_by_url(SHEET_URL)
    return sh.worksheet(WORKSHEET_NAME)

def get_ai_summary(subject, score):

    try:
        api_key = userdata.get('gemini_key')
        url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?key={api_key}"

        # 根據分數判斷引導語
        if score < 60:
            tone_instruction = "學生此科目不及格，請用溫暖且極具鼓勵性的語氣，並提供具體的學習建議"
        else:
            tone_instruction = "學生表現良好，請給予肯定並建議如何保持優勢"

        prompt = f"你是導師，{tone_instruction}。科目：{subject}，成績：{score}分。"

        payload = {"contents": [{"parts": [{"text": prompt}]}]}
        headers = {'Content-Type': 'application/json'}

        response = requests.post(url, headers=headers, data=json.dumps(payload), timeout=10)

        if response.status_code == 200:
            return response.json()['candidates'][0]['content']['parts'][0]['text']
        else:
            # API 失敗時的備用邏輯
            if score < 60:
                return f"【AI分析】{subject}成績為 {score}。目前雖然暫時落後，但只要找到適合的讀書方法並補強基礎，一定能迎頭趕上，加油！"
            return f"【AI分析】{subject}成績為 {score}。表現相當不錯，繼續保持對學習的熱忱！"

    except Exception as e:
        return f"AI 分析發生錯誤: {e}"

def grade_manager_process(subject, score):
    """單科處理邏輯：寫入 Sheets 並獲取 AI 總結"""
    try:
        # 基礎檢查
        if not subject.strip():
            return "❌ 請輸入學科名稱", ""

        ws = get_sheet()
        today = datetime.now().strftime('%Y-%m-%d')

        # 1. 獲取 AI 總結
        ai_report = get_ai_summary(subject, score)

        # 2. 寫入 Google Sheets (第一行寫成績，第二行寫 AI 總結)
        # 格式：[日期, 科目/類別, 成績或內容]
        ws.append_row([today, subject, score])
        ws.append_row([today, f"🤖 AI分析建議", ai_report])

        # 3. 簡單美化最後兩行
        last_row = len(ws.col_values(1))
        ws.format(f"A{last_row}:C{last_row}", {
            "backgroundColor": {"red": 0.95, "green": 0.95, "blue": 1.0},
            "textFormat": {"italic": True}
        })

        return f"✅ {subject} 資料已成功紀錄！", ai_report

    except Exception as e:
        return f"❌ 發生錯誤：{str(e)}", ""

# --- Gradio 介面設計 ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎓 AI 成績管理助教")
    gr.Markdown("輸入學科與成績，AI 導師將為您分析並自動同步至 Google Sheets。")

    with gr.Row():
        with gr.Column():
            # 單一輸入型組件
            input_subject = gr.Textbox(
                label="學科名稱",
                placeholder="例如：國文",
                interactive=True
            )
            input_score = gr.Number(
                label="分數 (0-100)",
                value=60,
                precision=0,
                minimum=0,
                maximum=100
            )
            btn = gr.Button("🚀 提交並紀錄分析", variant="primary")

        with gr.Column():
            output_status = gr.Textbox(label="處理狀態", interactive=False)

            output_ai = gr.Markdown("等待輸入分析...")

    # 點擊按鈕後的行為
    btn.click(
        fn=grade_manager_process,
        inputs=[input_subject, input_score],
        outputs=[output_status, output_ai]
    )

# 啟動應用
if __name__ == "__main__":
    demo.launch(debug=True)

/tmp/ipykernel_10717/2982339239.py:87: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/04/01 15:02:28 [W] [service.go:132] login to server failed: tls: failed to verify certificate: x509: certificate has expired or is not yet valid: current time 2026-04-01T15:02:28Z is after 2026-04-01T07:01:35Z


<IPython.core.display.Javascript object>

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> None
